In [78]:
from collections.abc import Iterable, Iterator
from typing import Any, Final, Self

In [79]:
def balanced_modulo(n: int, d: int) -> int:
    """Return n % d wrapped to a range of values balanced around zero.

    Whereas n % d would normally produce a value in the range from 0 to d - 1,
    this function instead returns a value in the range from -(d - 1) / 2 up to
    (d - 1) / 2. Since the range is intended to be perfectly balanced around
    zero, a ValueError is thrown if d is an even integer.
    """
    if d < 0 or d % 2 != 1:
        raise ValueError('d must be a positive odd integer')
    h = (d - 1) // 2
    return (n + h) % d - h


# Test balanced_modulo() to make sure it's behaving as desired
for x in range(-5, 6):
    print(f'balanced_modulo({x}, 3) = {balanced_modulo(x, 3)}')    

balanced_modulo(-5, 3) = 1
balanced_modulo(-4, 3) = -1
balanced_modulo(-3, 3) = 0
balanced_modulo(-2, 3) = 1
balanced_modulo(-1, 3) = -1
balanced_modulo(0, 3) = 0
balanced_modulo(1, 3) = 1
balanced_modulo(2, 3) = -1
balanced_modulo(3, 3) = 0
balanced_modulo(4, 3) = 1
balanced_modulo(5, 3) = -1


In [80]:
TRIT_VALUES: Final[tuple[int, ...]] = (-1, 0, +1)


class Trit:

    __slots__ = ('value',)

    __match_args__ = ('value',)

    def __setattr__(self, _name: str, _value: Any) -> None:
        raise TypeError('trit objects are immutable')

    def __delattr__(self, _name: str) -> None:
        raise TypeError('trit objects are immutable')

    def __init__(self, value: int, force_wrap: bool = False) -> None:
        if force_wrap:
            value = balanced_modulo(value, len(TRIT_VALUES))
        elif value not in TRIT_VALUES:
            raise ValueError(f'invalid trit value: {value!r}')
        self.value: int
        object.__setattr__(self, 'value', int(value))
    
    def __repr__(self) -> str:
        value_str = '+1' if self.value == +1 else str(self.value)
        return f'{self.__class__.__name__}({value_str})'

    def __str__(self) -> str:
        return 'T' if self.value == -1 else str(self.value)

    def __pos__(self) -> Self:
        return self

    def __neg__(self) -> Self:
        return self.__class__(-self.value)

    def __add__(self, other: Any) -> Self:
        if isinstance(other, int):
            return self.__class__(self.value + other, force_wrap=True)
        if isinstance(other, self.__class__):
            return self + other.value
        return NotImplemented

    def __radd__(self, other: Any) -> Self:
        return self + other
    
    def __sub__(self, other: Any) -> Self:
        if isinstance(other, int):
            return self + -other
        if isinstance(other, self.__class__):
            return self + -other.value
        return NotImplemented

    def __rsub__(self, other: Any) -> Self:
        return -self + other

    def __invert__(self) -> Self:
        return -self
    
    def __and__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            return self.__class__(min(self.value, other.value))
        if isinstance(other, int):
            return self & self.__class__(other)
        return NotImplemented

    def __rand__(self, other: Any) -> Self:
        return self & other

    def __or__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            return self.__class__(max(self.value, other.value))
        if isinstance(other, int):
            return self | self.__class__(other)
        return NotImplemented

    def __ror__(self, other: Any) -> Self:
        return self | other

    def __xor__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            if 0 in (self.value, other.value):
                result_value = 0  # Unknown values propagate
            elif self.value == other.value:
                result_value = -1  # Thing xor thing -> false
            else:  # self.value != other.value
                result_value = +1  # Thing xor (not thing) -> true
        if isinstance(other, int):
            return self ^ self.__class__(other)
        return NotImplemented

    def __rxor__(self) -> Self:
        return self ^ other

    def carry_trit(self, other: Any) -> Self:
        if (self.value, other.value) == (-1, -1):
            return self.__class__(-1)
        if (self.value, other.value) == (+1, +1):
            return self.__class__(+1)
        return self.__class__(0)

    def __hash__(self) -> int:
        return hash(self.value)
    
    def __eq__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            return self.value == other.value
        return self.value == other

    def __ne__(self, other: Any) -> bool:
        return not self == other

    def __lt__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            return self.value < other.value
        return self.value < other

    def __gt__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            return self.value > other.value
        return self.value > other

    def __le__(self, other: Any) -> bool:
        return not self > other

    def __ge__(self, other: Any) -> bool:
        return not self < other

    def __abs__(self) -> Self:
        return self.__class__(abs(self.value))
    
    def __bool__(self) -> bool:
        return self.value != 0

    def __int__(self) -> int:
        return self.value

    def __float__(self) -> float:
        return float(self.value)

    def __complex__(self) -> complex:
        return complex(self.value)


# Test the Trit class to ensure it behaves as needed
for x in TRIT_VALUES:
    t = Trit(x)
    print(f'x = {x}')
    print(f'repr(t) = {repr(repr(t))}')
    print(f'str(t) = {repr(str(t))}')
    print(f'+t = {+t}')
    print(f'-t = {-t}')
    print(f't+1 = {t+1}')
    print(f'1+t = {1+t}')
    print(f't-1 = {t-1}')
    print(f'1-t = {1-t}')
    print(f'hash(t)==hash(x) {hash(t)==hash(x)}')
    print(f't==Trit(x) = {t==Trit(x)}')
    print(f't==x = {t==x}')
    print(f't<x+1 = {t<x+1}')
    print(f't>x-1 = {t>x-1}')
    print(f't<=x = {t<=x}')
    print(f't>=x = {t>=x}')
    print(f'bool(t) = {bool(t)}')
    print(f'int(t) = {int(t)}')
    print(f'float(t) = {float(t)}')
    print(f'complex(t) = {complex(t)}')
    try:
        t.value = 14
    except TypeError:
        print('setattr works')
    else:
        assert False, 'setattr did not raise an exception'
    match t:
        case Trit(-1):
            print('I\'m negative one!')
        case Trit(0):
            print('I\'m zero!')
        case Trit(1):
            print('I\'m positive one!')
        case _:
            assert False, 'match case did not work correctly'
    print()

x = -1
repr(t) = 'Trit(-1)'
str(t) = 'T'
+t = T
-t = 1
t+1 = 0
1+t = 0
t-1 = 1
1-t = T
hash(t)==hash(x) True
t==Trit(x) = True
t==x = True
t<x+1 = True
t>x-1 = True
t<=x = True
t>=x = True
bool(t) = True
int(t) = -1
float(t) = -1.0
complex(t) = (-1+0j)
setattr works
I'm negative one!

x = 0
repr(t) = 'Trit(0)'
str(t) = '0'
+t = 0
-t = 0
t+1 = 1
1+t = 1
t-1 = T
1-t = 1
hash(t)==hash(x) True
t==Trit(x) = True
t==x = True
t<x+1 = True
t>x-1 = True
t<=x = True
t>=x = True
bool(t) = False
int(t) = 0
float(t) = 0.0
complex(t) = 0j
setattr works
I'm zero!

x = 1
repr(t) = 'Trit(+1)'
str(t) = '1'
+t = 1
-t = T
t+1 = T
1+t = T
t-1 = 0
1-t = 0
hash(t)==hash(x) True
t==Trit(x) = True
t==x = True
t<x+1 = True
t>x-1 = True
t<=x = True
t>=x = True
bool(t) = True
int(t) = 1
float(t) = 1.0
complex(t) = (1+0j)
setattr works
I'm positive one!



In [85]:
TRYTE_SIZE: Final[int] = 5
TRYTE_TRIT_WEIGHTS: Final[tuple[int, ...]] = tuple(
    len(TRIT_VALUES) ** (TRYTE_SIZE - 1 - i) for i in range(TRYTE_SIZE)
)


class Tryte:

    __slots__ = ('trits',)

    def __setattr__(self, _name: str, _value: Any) -> None:
        raise TypeError('tryte objects are immutable')

    def __delattr__(self, _name: str) -> None:
        raise TypeError('tryte objects are immutable')

    def __init__(self, trits: Iterable[int | Trit]) -> None:
        trits_tuple = tuple(
            (t if isinstance(t, Trit) else Trit(t)) for t in trits
        )
        if len(trits_tuple) != TRYTE_SIZE:
            raise TypeError('trytes must be exactly {TRYTE_SIZE} trits')
        self.trits: tuple[int, ...]
        object.__setattr__(self, 'trits', trits_tuple)

    def __repr__(self) -> str:
        trits_string = ', '.join(
            ('+1' if t == +1 else str(int(t))) for t in self.trits
        )
        return f'{self.__class__.__name__}([{trits_string}])'

    def __str__(self) -> str:
        return ''.join(str(t) for t in self.trits)

    def __int__(self) -> int:
        return sum(
            int(t) * weight for t, weight
            in zip(self.trits, TRYTE_TRIT_WEIGHTS, strict=True)
        )

    @classmethod
    def from_int(cls, n: int, force_wrap: bool = False) -> Self:
        if force_wrap:
            n = balanced_modulo(n, len(TRIT_VALUES) ** TRYTE_SIZE)
        trit_values: list[int] = []
        left_over = n
        while left_over != 0:
            trit_value = balanced_modulo(left_over, len(TRIT_VALUES))
            trit_values.append(trit_value)
            left_over = (left_over - trit_value) // len(TRIT_VALUES)
        if len(trit_values) > TRYTE_SIZE:
            raise TypeError(f'int {n} cannot fit into one tryte')
        while len(trit_values) < TRYTE_SIZE:
            trits.append(0)
        return cls(reversed(trit_values))

    @classmethod
    def minimum(cls) -> Self:
        return cls(min(TRIT_VALUES) for _ in range(TRYTE_SIZE))

    @classmethod
    def zero(cls) -> Self:
        return cls(0 for _ in range(TRYTE_SIZE))
    
    @classmethod
    def maximum(cls) -> Self:
        return cls(max(TRIT_VALUES) for _ in range(TRYTE_SIZE))
    
    def __len__(self) -> int:
        return len(self.trits)

    def __iter__(self) -> Iterator[Trit]:
        return iter(self.trits)

    def __reversed__(self) -> Iterator[Trit]:
        return iter(reversed(self.trits))
    
    def __getitem__(self, index: int) -> Trit:
        return self.trits[index]

    def __contains__(self, trit: int | Trit) -> bool:
        for t in self.trits:
            if t == trit:
                return True
        return False

    def zip_trits(self, other: Self) -> Iterator[tuple[Trit, Trit]]:
        return zip(self.trits, other.trits, strict=True)
    
    def __pos__(self) -> Self:
        return self

    def __neg__(self) -> Self:
        return self.__class__(-t for t in self.trits)

    def __invert__(self) -> Self:
        return self.__class__(~t for t in self.trits)
    
    def __and__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            return self.__class__(
                self_t & other_t for self_t, other_t in self.zip_trits(other)
            )
        if isinstance(other, int):
            return self & self.__class__.from_int(other)
        return NotImplemented

    def __rand__(self, other: Any) -> Self:
        return self & other

    def __or__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            return self.__class__(
                self_t | other_t for self_t, other_t in self.zip_trits(other)
            )
        if isinstance(other, int):
            return self | self.__class__.from_int(other)
        return NotImplemented

    def __ror__(self, other: Any) -> Self:
        return self | other

    def __xor__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            return self.__class__(
                self_t ^ other_t for self_t, other_t in self.zip_trits(other)
            )
        if isinstance(other, int):
            return self | self.__class__.from_int(other)
        return NotImplemented

    def __rxor__(self, other: Any) -> Self:
        return self ^ other

    def __rshift__(self, other: Any) -> Self:
        if isinstance(other, int):
            if other < 0:
                raise ValueError('negative shift amount')
            if other >= TRYTE_SIZE:
                return self.__class__.zero()
            surviving_trits = list(self.trits[:TRYTE_SIZE-other])
            return self.__class__([0] * other + surviving_trits)
        if isinstance(other, self.__class__):
            return self >> int(other)
        return NotImplemented

    def __rrshift__(self, other: Any) -> Self:
        return self.__class__(other) >> self

    def __lshift__(self, other: Any) -> Self:
        if isinstance(other, int):
            if other < 0:
                raise ValueError('negative shift amount')
            if other >= TRYTE_SIZE:
                return self.__class__.zero()
            surviving_trits = list(self.trits[other:])
            return self.__class__(surviving_trits + [0] * other)
        if isinstance(other, self.__class__):
            return self << int(other)
        return NotImplemented
    
    def __rlshift__(self, other: Any) -> Self:
        return self.__class__(other) << self
    
    def __add__(self, other: Any) -> Self:
        if isinstance(other, self.__class__):
            basic_sum = self.__class__(
                self_t + other_t for self_t, other_t in self.zip_trits(other)
            )
            carry = self.__class__(
                self_t.carry_trit(other_t)
                for self_t, other_t in self.zip_trits(other)
            )
            if carry == 0:
                return basic_sum
            return basic_sum + (carry << 1)
        if isinstance(other, int):
            return self + self.__class__.from_int(other)
        return NotImplemented

    def __radd__(self, other: Any) -> Self:
        return self + other

    def __sub__(self, other: Any) -> Self:
        return self + -other

    def __rsub__(self, other: Any) -> Self:
        return -self + other

    def __mul__(self, other: Any) -> Self:
        # TODO: improve this implementation
        if isinstance(other, int):
            if other < 0:
                return -self * -other
            accumulator = self.__class__.zero()
            for _ in range(other):
                accumulator += self
            return accumulator
        if isinstance(other, self.__class__):
            return self * int(other)
        return NotImplemented

    def __rmul__(self, other: Any) -> Self:
        return self * other

    def __floordiv__(self, other: Any) -> Self:
        # TODO: improve this implementation
        if isinstance(other, int):
            return self.__class__(int(self) // other)
        if isinstance(other, self.__class__):
            return self // int(other)
        return NotImplemented

    def __rfloordiv__(self, other: Any) -> Self:
        return self.__class__(other) // self
    
    def __mod__(self, other: Any) -> Self:
        # TODO: improve this implementation
        if isinstance(other, int):
            return self.__class__(int(self) % other)
        if isinstance(other, self.__class__):
            return self % int(other)
        return NotImplemented

    def __rmod__(self, other: Any) -> Self:
        return self.__class__(other) % self

    def __divmod__(self, other: Any) -> tuple[Self, Self]:
        return self // other, self % other

    def __rdivmod__(self, other: Any) -> tuple[Self, Self]:
        return self.__class__(other) // self, self.__class(other) % self

    def __pow__(self, other: Any) -> Self:
        if isinstance(other, int):
            if other < 0:
                raise ValueError('negative exponent')
            if other == 0:
                return self.__class__.zero() + 1
            if self in (0, 1):
                return self
            accumulator = self.__class__.zero() + 1
            for _ in range(other):
                accumulator *= self
            return accumulator
        if isinstance(other, self.__class__):
            return self ** int(other)
        return NotImplemented

    def __rpow__(self, other: Any) -> Self:
        return self.__class__(other) ** self

    def __hash__(self) -> int:
        return hash(int(self))

    def __eq__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            return self.trits == other.trits
        return int(self) == other

    def __ne__(self, other: Any) -> bool:
        return not self == other

    def __lt__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            for self_t, other_t in self.zip_trits(other):
                if self_t != other_t:
                    return self_t < other_t
            return False  # self == other
        return int(self) < other

    def __gt__(self, other: Any) -> bool:
        if isinstance(other, self.__class__):
            for self_t, other_t in self.zip_trits(other):
                if self_t != other_t:
                    return self_t > other_t
        return int(self) > other

    def __le__(self, other: Any) -> bool:
        return not self > other

    def __ge__(self, other: Any) -> bool:
        return not self < other

    def __abs__(self) -> Self:
        return -self if self < 0 else self

    def __index__(self) -> int:
        return int(self)
    
    def __bool__(self) -> bool:
        return any(bool(t) for t in self.trits)

    def __float__(self) -> float:
        return float(int(self))

    def __complex__(self) -> complex:
        return complex(int(self))


print(TRYTE_TRIT_WEIGHTS)

print()

t = Tryte([+1, -1, 0, -1, +1])
print(t, repr(t))
print(int(t))
print(Tryte.from_int(52))

print()

t = Tryte([-1, -1, +1, 0, +1])
print(t, repr(t))
print(int(t))
print(Tryte.from_int(-98))

print()

t = Tryte([+1, -1, 0, 0, 0])
print(t, int(t))
print(t << 1, int(t << 1))
print((81-27), balanced_modulo((81-27)*3, 3**5))

(81, 27, 9, 3, 1)

1T0T1 Tryte([+1, -1, 0, -1, +1])
52
1T0T1

TT101 Tryte([-1, -1, +1, 0, +1])
-98
TT101

1T000 54
T0000 -81
54 -81
